# 05: Cross-file validation

Now we have **8 files** from multiple acquisition sessions. The critical experiment: **train on files from one session, test on files from another**. If the signal is real biology, it should generalize. If it's batch, it won't.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, roc_auc_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


## 1. Load t=0 cells from all files

In [ ]:
DATA_DIR = Path('/baldig/bioprojects2/emartinl/chemores/preprocessed_phasor')

# All preprocessed files with session labels
FILE_INFO = [
    # name              group              session
    ('CNTL-MB231',     'Control',         'session_orig'),
    ('TAMO-MB231',     'Chemoresistant',  'session_orig'),
    ('CNTL_75uM_p1',  'Control',         'session_75uM'),
    ('CNTL_75uM_p2',  'Control',         'session_75uM'),
    ('CNTL_75uM_p3',  'Control',         'session_75uM'),
    ('CNTL_75uM_p4',  'Control',         'session_75uM'),
    ('TAMO_p1',       'Chemoresistant',  'session_tamo_new'),
    ('TAMO_p2',       'Chemoresistant',  'session_tamo_new'),
]

# Load only t=0 cells from each file (memory-mapped)
all_cells = []
all_meta = []

for name, group, session in FILE_INFO:
    meta = pd.read_csv(DATA_DIR / f'{name}_cells_meta.csv')
    t0_mask = meta['time'] == 0
    t0_idx = np.where(t0_mask)[0]
    
    X = np.load(DATA_DIR / f'{name}_cells.npy', mmap_mode='r')
    cells_t0 = X[t0_idx]  # still memory-mapped slice
    
    meta_t0 = meta.loc[t0_mask].copy()
    meta_t0['file'] = name
    meta_t0['session'] = session
    # reassign cell_id to be a global index
    meta_t0 = meta_t0.reset_index(drop=True)
    
    all_cells.append(np.array(cells_t0))  # load into RAM
    all_meta.append(meta_t0)
    print(f'{name:20s}  {group:16s}  {session:18s}  t0 cells: {len(t0_idx)}')

X_all = np.concatenate(all_cells, axis=0)
meta_all = pd.concat(all_meta, ignore_index=True)
meta_all['global_id'] = np.arange(len(meta_all))

print(f'\nTotal t=0 cells: {len(meta_all)}')
print(f'X_all shape: {X_all.shape}')
print(f'RAM: ~{X_all.nbytes / 1e9:.1f} GB')

CNTL-MB231            Control           session_orig        t0 cells: 325
TAMO-MB231            Chemoresistant    session_orig        t0 cells: 350
CNTL                  Control           session_testset     t0 cells: 493
CNTL_2                Control           session_testset     t0 cells: 253
CNTL_75uM_p1          Control           session_75uM        t0 cells: 180
CNTL_75uM_p2          Control           session_75uM        t0 cells: 90
CNTL_75uM_p3          Control           session_75uM        t0 cells: 144
CNTL_75uM_p4          Control           session_75uM        t0 cells: 126
TAMO_p1               Chemoresistant    session_tamo_new    t0 cells: 425
TAMO_p2               Chemoresistant    session_tamo_new    t0 cells: 725

Total t=0 cells: 3111
X_all shape: (3111, 7, 512, 512)
RAM: ~45.7 GB


In [3]:
# Summary table
summary = meta_all.groupby(['session', 'file', 'group']).size().reset_index(name='n_cells')
print(summary.to_string(index=False))
print()
print('Per group:')
print(meta_all.groupby('group').size())
print()
print('Per session:')
print(meta_all.groupby(['session', 'group']).size().unstack(fill_value=0))

         session         file          group  n_cells
    session_75uM CNTL_75uM_p1        Control      180
    session_75uM CNTL_75uM_p2        Control       90
    session_75uM CNTL_75uM_p3        Control      144
    session_75uM CNTL_75uM_p4        Control      126
    session_orig   CNTL-MB231        Control      325
    session_orig   TAMO-MB231 Chemoresistant      350
session_tamo_new      TAMO_p1 Chemoresistant      425
session_tamo_new      TAMO_p2 Chemoresistant      725
 session_testset         CNTL        Control      493
 session_testset       CNTL_2        Control      253

Per group:
group
Chemoresistant    1500
Control           1611
dtype: int64

Per session:
group             Chemoresistant  Control
session                                  
session_75uM                   0      540
session_orig                 350      325
session_tamo_new            1150        0
session_testset                0      746


## 2. Model and training utilities

In [4]:
class CellDataset(Dataset):
    def __init__(self, X, labels):
        self.X = torch.from_numpy(np.ascontiguousarray(X)).float()
        self.y = torch.from_numpy(np.asarray(labels)).long()
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class SmallCNN(nn.Module):
    """Same architecture as in 04_t0_sanity_check."""
    def __init__(self, in_ch=7):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_ch, 16, 5, stride=2, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def train_and_eval(X_train, y_train, X_test, y_test,
                   epochs=10, lr=1e-3, batch_size=16, verbose=True):
    """Train SmallCNN and return test accuracy + AUC."""
    train_ds = CellDataset(X_train, y_train)
    test_ds = CellDataset(X_test, y_test)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_dl = DataLoader(test_ds, batch_size=batch_size)

    model = SmallCNN().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            opt.step()

    # Evaluate
    model.eval()
    all_preds, all_probs, all_labels = [], [], []
    with torch.no_grad():
        for xb, yb in test_dl:
            xb = xb.to(device)
            logits = model(xb)
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            preds = logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_probs.extend(probs)
            all_labels.extend(yb.numpy())

    acc = accuracy_score(all_labels, all_preds)
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        auc = float('nan')

    if verbose:
        print(f'  Test acc: {acc:.3f}  AUC: {auc:.3f}  '
              f'(train={len(y_train)}, test={len(y_test)})')
    return acc, auc

## 3. Experiment A: within-file baseline (sanity check of the replicated results)

In [5]:
# Original files only
orig_mask = meta_all['file'].isin(['CNTL-MB231', 'TAMO-MB231'])
meta_orig = meta_all[orig_mask]
X_orig = X_all[meta_orig['global_id'].values]
y_orig = (meta_orig['group'] == 'Chemoresistant').astype(int).values

# 80/20 random split
np.random.seed(42)
n = len(y_orig)
idx = np.random.permutation(n)
split = int(0.8 * n)

print('Exp A: Within-file (original 2 files), random 80/20 split, t=0')
acc_a, auc_a = train_and_eval(
    X_orig[idx[:split]], y_orig[idx[:split]],
    X_orig[idx[split:]], y_orig[idx[split:]],
    epochs=10
)

Exp A: Within-file (original 2 files), random 80/20 split, t=0
  Test acc: 1.000  AUC: 1.000  (train=540, test=135)


## 4. Experiment B: cross-session generalization (the key test)

**Train on original session** (CNTL-MB231 + TAMO-MB231), **test on new sessions**.

In [6]:
# --- Train: original session ---
train_mask = meta_all['session'] == 'session_orig'
meta_train = meta_all[train_mask]
X_train = X_all[meta_train['global_id'].values]
y_train = (meta_train['group'] == 'Chemoresistant').astype(int).values

# --- Test: all new files ---
test_mask = ~train_mask
meta_test = meta_all[test_mask]
X_test = X_all[meta_test['global_id'].values]
y_test = (meta_test['group'] == 'Chemoresistant').astype(int).values

print('Exp B: Train on original session → Test on all new sessions')
print(f'  Train: {len(y_train)} cells ({y_train.sum()} chemo, {(1-y_train).sum()} ctrl)')
print(f'  Test:  {len(y_test)} cells ({y_test.sum()} chemo, {(1-y_test).sum()} ctrl)')
acc_b, auc_b = train_and_eval(X_train, y_train, X_test, y_test, epochs=10)

Exp B: Train on original session → Test on all new sessions
  Train: 675 cells (350 chemo, 325 ctrl)
  Test:  2436 cells (1150 chemo, 1286 ctrl)
  Test acc: 0.495  AUC: 0.436  (train=675, test=2436)


## 5. Experiment C: reverse direction

**Train on new sessions** (75uM + TAMO_new + testset), **test on original session**.

In [7]:
# Reverse: train on new, test on original
print('Exp C: Train on new sessions → Test on original session')
print(f'  Train: {len(y_test)} cells')
print(f'  Test:  {len(y_train)} cells')
acc_c, auc_c = train_and_eval(X_test, y_test, X_train, y_train, epochs=10)

Exp C: Train on new sessions → Test on original session
  Train: 2436 cells
  Test:  675 cells
  Test acc: 0.519  AUC: 0.192  (train=2436, test=675)


## 6. Experiment D: new files only (within new sessions)

Train and test only on the new files (random 80/20 split). This tells us if the new data alone shows a signal.

In [8]:
# New files only
new_mask = meta_all['session'] != 'session_orig'
meta_new = meta_all[new_mask]
X_new = X_all[meta_new['global_id'].values]
y_new = (meta_new['group'] == 'Chemoresistant').astype(int).values

np.random.seed(42)
n = len(y_new)
idx = np.random.permutation(n)
split = int(0.8 * n)

print('Exp D: New files only, random 80/20 split')
print(f'  Total: {n} cells ({y_new.sum()} chemo, {(1-y_new).sum()} ctrl)')
acc_d, auc_d = train_and_eval(
    X_new[idx[:split]], y_new[idx[:split]],
    X_new[idx[split:]], y_new[idx[split:]],
    epochs=10
)

Exp D: New files only, random 80/20 split
  Total: 2436 cells (1150 chemo, 1286 ctrl)
  Test acc: 0.986  AUC: 1.000  (train=1948, test=488)


## 7. Experiment E: leave-one-file-out (new files)

For the new multi-timepoint files, hold out one file at a time and train on the rest. This shows consistency of the signal.

In [11]:
# Only multi-tp new files (exclude CNTL and CNTL_2 which are single-tp with few cells)
lofo_files = ['CNTL_75uM_p1', 'CNTL_75uM_p2', 'CNTL_75uM_p3', 'CNTL_75uM_p4',
              'TAMO_p1', 'TAMO_p2']

lofo_mask = meta_all['file'].isin(lofo_files)
meta_lofo = meta_all[lofo_mask]
X_lofo = X_all[meta_lofo['global_id'].values]
y_lofo = (meta_lofo['group'] == 'Chemoresistant').astype(int).values
files_lofo = meta_lofo['file'].values

print('Exp E: Leave-one-file-out (new multi-tp files only)')
print('  (AUC undefined — each held-out file has only one class)\n')
results_e = []
for held_out in lofo_files:
    test_idx = np.where(files_lofo == held_out)[0]
    train_idx = np.where(files_lofo != held_out)[0]
    
    if len(np.unique(y_lofo[train_idx])) < 2:
        print(f'  Hold out {held_out}: SKIP (only one class in train)')
        continue
    
    print(f'  Hold out: {held_out} ({len(test_idx)} cells)', end='')
    acc, auc = train_and_eval(
        X_lofo[train_idx], y_lofo[train_idx],
        X_lofo[test_idx], y_lofo[test_idx],
        epochs=10, verbose=False
    )
    print(f'  → acc={acc:.3f}')
    results_e.append({'held_out': held_out, 'acc': acc, 'n_test': len(test_idx)})

df_e = pd.DataFrame(results_e)
print(f'\nMean acc: {df_e["acc"].mean():.3f} ± {df_e["acc"].std():.3f}')

Exp E: Leave-one-file-out (new multi-tp files only)
  (AUC undefined — each held-out file has only one class)

  Hold out: CNTL_75uM_p1 (180 cells)  → acc=0.450
  Hold out: CNTL_75uM_p2 (90 cells)  → acc=0.900
  Hold out: CNTL_75uM_p3 (144 cells)  → acc=0.562
  Hold out: CNTL_75uM_p4 (126 cells)  → acc=1.000
  Hold out: TAMO_p1 (425 cells)  → acc=0.294
  Hold out: TAMO_p2 (725 cells)  → acc=0.414

Mean acc: 0.603 ± 0.284


## 8. Summary

In [14]:
summary_results = pd.DataFrame([
    {'Experiment': 'A: Within-file (orig, random split)', 'Acc': acc_a, 'AUC': auc_a},
    {'Experiment': 'B: Train orig → Test new',            'Acc': acc_b, 'AUC': auc_b},
    {'Experiment': 'C: Train new → Test orig',            'Acc': acc_c, 'AUC': auc_c},
    {'Experiment': 'D: Within new files (random split)',   'Acc': acc_d, 'AUC': auc_d},
])

if len(results_e) > 0:
    summary_results = pd.concat([summary_results, pd.DataFrame([{
        'Experiment': 'E: LOFO (mean ± std)',
        'Acc': df_e['acc'].mean(),
        'AUC': float('nan'),
    }])], ignore_index=True)

print(summary_results.to_string(index=False, float_format='%.3f'))
print()
print('Interpretation:')
print('  A = 100%  → batch effect confirmed (identical to NB04)')
print('  B =  50%  → signal does NOT generalize across sessions')
print('  C =  52%  → same in reverse — no transferable biology')
print('  D =  99%  → new files also have batch effect (CNTL_75uM vs TAMO sessions)')
print('  E =  62%  → unstable across files, high variance (0.29–1.00)')
print()
print('Conclusion: all high accuracy is driven by batch/session identity, not biology.')
print('A proper study needs matched Control and Chemo from the SAME acquisition session.')

                         Experiment   Acc   AUC
A: Within-file (orig, random split) 1.000 1.000
           B: Train orig → Test new 0.495 0.436
           C: Train new → Test orig 0.519 0.192
 D: Within new files (random split) 0.986 1.000
               E: LOFO (mean ± std) 0.603   NaN

Interpretation:
  A = 100%  → batch effect confirmed (identical to NB04)
  B =  50%  → signal does NOT generalize across sessions
  C =  52%  → same in reverse — no transferable biology
  D =  99%  → new files also have batch effect (CNTL_75uM vs TAMO sessions)
  E =  62%  → unstable across files, high variance (0.29–1.00)

Conclusion: all high accuracy is driven by batch/session identity, not biology.
A proper study needs matched Control and Chemo from the SAME acquisition session.
